In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import spherical_jn, factorial
from scipy.integrate import cumulative_trapezoid

# =====================================================
# Sulfur Hartree-Fock STO data
# S: 1s2 2s2 2p6 3s2 3p4
# Ground term: 3P
# Format: (n_j, Z_j, C_j)
# =====================================================

sulfur_1s = [
    (1, 22.3787,  0.367468),
    (1, 13.7956,  0.471256),
    (2, 19.9766,  0.192030),
    (2,  7.5145,  0.000539),
    (2,  5.7222,  0.002074),
    (2,  4.2264, -0.0000864),
    (3, 42.1787, -0.000442),
    (3,  3.3088,  0.000163),
    (3,  2.1707, -0.000073),
    (3,  1.5040,  0.000014),
]

sulfur_2s = [
    (1, 22.3787,  0.070509),
    (1, 13.7956, -0.492518),
    (2, 19.9766,  0.056472),
    (2,  7.5145,  0.114779),
    (2,  5.7222,  0.846899),
    (2,  4.2264,  0.150553),
    (3, 42.1787,  0.000233),
    (3,  3.3088, -0.002483),
    (3,  2.1707,  0.001580),
    (3,  1.5040,  0.000045),
]

sulfur_3s = [
    (1, 22.3787,  0.028833),
    (1, 13.7956, -0.160596),
    (2, 19.9766,  0.022476),
    (2,  7.5145,  0.056607),
    (2,  5.7222,  0.107272),
    (2,  4.2264,  0.326955),
    (3, 42.1787,  0.000067),
    (3,  3.3088, -0.337632),
    (3,  2.1707, -0.602710),
    (3,  1.5040, -0.272978),
]

sulfur_2p = [
    (2, 22.6414, -0.000466),
    (2, 10.4197,  0.141231),
    (2,  6.1160,  0.501894),
    (2,  4.4156,  0.403324),
    (3, 17.3448, -0.006509),
    (3,  2.6496,  0.004375),
    (3,  1.6975,  0.000225),
    (3,  1.1477,  0.000315),
]

sulfur_3p = [
   (2, 22.6414,  0.001263),
   (2, 10.4197, -0.047039),
   (2,  6.1160, -0.090305),
   (2,  4.4156, -0.159888),
   (3, 17.3448,  0.005017),
   (3,  2.6496,  0.341230),
   (3,  1.6975,  0.519259),
   (3,  1.1477,  0.262504),
]
# =====================================================
# HF normalization constant
# N_j = sqrt((2Z)^(2n+1)/(2n)!)
# =====================================================
def N_sto(n, Z):
    return np.sqrt((2*Z)**(2*n + 1) / factorial(2*n, exact=False))

def R_nl(r, coeffs):
    R = np.zeros_like(r)
    for n, Z, C in coeffs:
        N = N_sto(n, Z)
        R += C * N * r**(n-1) * np.exp(-Z*r)
    return R
# =====================================================
# Grids
# =====================================================

r = np.linspace(1e-6, 60.0, 20000)     # radial grid
p = np.arange(0.0, 1000.0 + 0.05, 0.05) # momentum grid
Q = np.arange(0.0, 100.0 + 0.05, 0.05) # Q grid

# =====================================================
# Calculate chi_nl(p)
# chi_nl(p) = sqrt(2/pi) int R_nl(r) j_l(pr) r^2 dr
# =====================================================

def chi_p(p_grid, r_grid, R_grid, l):
    chi = []

    for pp in p_grid:
        jl = spherical_jn(l, pp*r_grid)
        integrand = R_grid * jl * r_grid**2
        val = np.sqrt(2/np.pi) * np.trapz(integrand, r_grid)
        chi.append(val)

    return np.array(chi)

# =====================================================
# Calculate J(Q)
# I(p) = |chi(p)|^2 p^2
# J(Q) = 1/2 int_Q^inf I(p)/p dp
# =====================================================

def calculate_J(p_grid, chi_grid, Q_grid):
    I = chi_grid**2 * p_grid**2

    integrand = np.zeros_like(p_grid)
    integrand[1:] = I[1:] / p_grid[1:]

    # Reverse cumulative integral from p to infinity
    rev_integral = cumulative_trapezoid(
        integrand[::-1],
        p_grid[::-1],
        initial=0
    )

    J_p = -0.5 * rev_integral[::-1]

    # Interpolate J from p-grid to Q-grid
    J_Q = np.interp(Q_grid, p_grid, J_p)

    return I, J_Q

# =====================================================
# Build Krypton radial orbitals
# =====================================================
def print_R_formula(coeffs, name):
    print(f"\n{name}(r) =")

    for n, Z, C in coeffs:
        N = N_sto(n, Z)
        A = C * N

        if n == 1:
            print(f"{A:+.4f} * exp(-{Z:.4f} r)")
        elif n == 2:
            print(f"{A:+.4f} * r * exp(-{Z:.4f} r)")
        elif n == 3:
            print(f"{A:+.4f} * r^2 * exp(-{Z:.4f} r)")
        elif n == 4:
            print(f"{A:+.4f} * r^3 * exp(-{Z:.4f} r)")
            
R1s = R_nl(r, sulfur_1s)
R2s = R_nl(r, sulfur_2s)
R3s = R_nl(r, sulfur_3s)

R2p = R_nl(r, sulfur_2p)
R3p = R_nl(r, sulfur_3p)

print_R_formula(sulfur_1s, "S_R1s")
print_R_formula(sulfur_2s, "S_R2s")
print_R_formula(sulfur_3s, "S_R3s")

print_R_formula(sulfur_2p, "S_R2p")
print_R_formula(sulfur_3p, "S_R3p")

# =====================================================
# Momentum-space wavefunctions
# =====================================================

chi_1s = chi_p(p, r, R1s, l=0)
chi_2s = chi_p(p, r, R2s, l=0)
chi_3s = chi_p(p, r, R3s, l=0)

chi_2p = chi_p(p, r, R2p, l=1)
chi_3p = chi_p(p, r, R3p, l=1)

# =====================================================
# Momentum densities and Compton profiles
# =====================================================
I_1s, J_1s = calculate_J(p, chi_1s, Q)
I_2s, J_2s = calculate_J(p, chi_2s, Q)
I_3s, J_3s = calculate_J(p, chi_3s, Q)

I_2p, J_2p = calculate_J(p, chi_2p, Q)
I_3p, J_3p = calculate_J(p, chi_3p, Q)

print("Norm 1s =", np.trapezoid(I_1s, p))
print("Norm 2s =", np.trapezoid(I_2s, p))
print("Norm 2p =", np.trapezoid(I_2p, p))
print("Norm 3s =", np.trapezoid(I_3s, p))
print("Norm 3p =", np.trapezoid(I_3p, p))

# S: 1s2 2s2 2p6 3s2 3p4
J_total = 2*J_1s + 2*J_2s + 6*J_2p + 2*J_3s + 4*J_3p 

# =====================================================
# Print and save table
# =====================================================
table = pd.DataFrame({
    "Q": Q,
    "J_1s_one_electron": J_1s,
    "J_2s_one_electron": J_2s,
    "J_3s_one_electron": J_3s,
    "J_2p_one_electron": J_2p,
    "J_3p_one_electron": J_3p,
    "J_total_S": J_total
    })
print(table)
table.to_csv("S_STO_HF_profile.csv", index = False)

# =====================================================
# Plot radial orbitals
# =====================================================

plt.figure(figsize=(8,6))
plt.plot(r, R1s, label="R_1s")
plt.plot(r, R2s, label="R_2s")
plt.plot(r, R2p, label="R_2p")
plt.plot(r, R3s, label="R_3s")
plt.plot(r, R3p, label="R_3p")
plt.xlim(0, 2)
plt.xlabel("r (a.u.)")
plt.ylabel("R_nl(r)")
plt.title("S Hartree-Fock Radial Orbitals")
plt.grid(True)
plt.legend()
plt.show()

# =====================================================
# Plot Compton profiles
# =====================================================
plt.figure(figsize=(8,6))
plt.plot(Q, 2*J_1s, label="1s(2)")
plt.plot(Q, 2*J_2s, label="2s(2)")
plt.plot(Q, 6*J_2p, label="2p(6)")
plt.plot(Q, 2*J_3s, label="3s(2)")
plt.plot(Q, 4*J_3p, label="3p(4)")

plt.plot(Q, J_total, label="S total", linewidth=2)

plt.xlabel("Q (a.u.)")
plt.ylabel("J(Q)")
plt.title(" S Compton Profile from HF STO Orbitals")
plt.yscale("log")
plt.grid(True)
plt.legend()
plt.show()


S_R1s(r) =
+77.8040 * exp(-22.3787 r)
+48.2945 * exp(-13.7956 r)
+395.4963 * r * exp(-19.9766 r)
+0.0963 * r * exp(-7.5145 r)
+0.1876 * r * exp(-5.7222 r)
-0.0037 * r * exp(-4.2264 r)
-90.8212 * r^2 * exp(-42.1787 r)
+0.0045 * r^2 * exp(-3.3088 r)
-0.0005 * r^2 * exp(-2.1707 r)
+0.0000 * r^2 * exp(-1.5040 r)

S_R2s(r) =
+14.9289 * exp(-22.3787 r)
-50.4735 * exp(-13.7956 r)
+116.3072 * r * exp(-19.9766 r)
+20.5155 * r * exp(-7.5145 r)
+76.5964 * r * exp(-5.7222 r)
+6.3839 * r * exp(-4.2264 r)
+47.8763 * r^2 * exp(-42.1787 r)
-0.0690 * r^2 * exp(-3.3088 r)
+0.0100 * r^2 * exp(-2.1707 r)
+0.0001 * r^2 * exp(-1.5040 r)

S_R3s(r) =
+6.1048 * exp(-22.3787 r)
-16.4579 * exp(-13.7956 r)
+46.2905 * r * exp(-19.9766 r)
+10.1179 * r * exp(-7.5145 r)
+9.7020 * r * exp(-5.7222 r)
+13.8638 * r * exp(-4.2264 r)
+13.7670 * r^2 * exp(-42.1787 r)
-9.3806 * r^2 * exp(-3.3088 r)
-3.8295 * r^2 * exp(-2.1707 r)
-0.4802 * r^2 * exp(-1.5040 r)

S_R2p(r) =
-1.3125 * r * exp(-22.6414 r)
+57.1528 * r * exp(-10.

/tmp/ipykernel_3953/610708075.py:106: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  val = np.sqrt(2/np.pi) * np.trapz(integrand, r_grid)


Norm 1s = 1.0034203641956712
Norm 2s = 1.0070840117813111
Norm 2p = 1.0000003171689886
Norm 3s = 0.9990939568775582
Norm 3p = 1.0000011865856335
           Q  J_1s_one_electron  J_2s_one_electron  J_3s_one_electron  \
0       0.00       5.473938e-02       2.314537e-01       7.940360e-01   
1       0.05       5.473761e-02       2.313303e-01       7.887810e-01   
2       0.10       5.473227e-02       2.309607e-01       7.732520e-01   
3       0.15       5.472339e-02       2.303463e-01       7.481373e-01   
4       0.20       5.471096e-02       2.294895e-01       7.145200e-01   
...      ...                ...                ...                ...   
1996   99.80       8.055573e-07       6.881858e-08       5.900133e-09   
1997   99.85       8.032005e-07       6.861590e-08       5.882754e-09   
1998   99.90       8.008517e-07       6.841391e-08       5.865435e-09   
1999   99.95       7.985108e-07       6.821261e-08       5.848175e-09   
2000  100.00       7.961779e-07       6.801201e-08  

PermissionError: [Errno 13] Permission denied: 'S_STO_HF_profile.csv'